# Bitcoin application of the SHA reverse tension probe

This notebook applies the **geometry-only tension probe** to **real Bitcoin mainnet block headers**.

What is tested here:

1. **Real Bitcoin data**  
   - **Genesis block** (block 0)  
   - **Mainnet block 328,734** (the official Bitcoin developer reference header example)

2. **Real SHA pathway**  
   Bitcoin block hashes use **double SHA-256** over an **80-byte block header**.  
   The 80-byte header forces the **first SHA-256 pass** to use **two compression blocks**:
   - block 1: first 64 bytes of the header
   - block 2: last 16 bytes of the header plus padding

3. **Mining-relevant location**  
   In the **second compression block of the first SHA pass**, the first four 32-bit seed words are:
   - `W[0]` = final 4 bytes of merkle root
   - `W[1]` = timestamp
   - `W[2]` = bits
   - `W[3]` = nonce

The notebook keeps the proof boundary explicit:

- **proved by direct computation here**
  - the header serializations
  - the double-SHA block hashes
  - the one-round reverse-step probe mechanics
  - the observed hot/cold ranking behavior on real block data

- **not proved here**
  - full inversion of Bitcoin proof-of-work
  - uniqueness from a one-round local tension score


In [1]:
from __future__ import annotations

from dataclasses import dataclass
import hashlib
import random
import statistics
import struct
from typing import Dict, List, Tuple

MASK32 = 0xFFFFFFFF

H0 = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

K = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2,
]

def u32(x: int) -> int:
    return x & MASK32

def rotr(x: int, n: int) -> int:
    x &= MASK32
    return ((x >> n) | (x << (32 - n))) & MASK32

def ch(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (~x & z)) & MASK32

def maj(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (x & z) ^ (y & z)) & MASK32

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)

def hw(x: int) -> int:
    return (x & MASK32).bit_count()

def add32(a: int, b: int) -> Tuple[int, int]:
    total = (a & MASK32) + (b & MASK32)
    return total & MASK32, int(total >> 32)

def carry_mask_add(a: int, b: int) -> int:
    a &= MASK32
    b &= MASK32
    carry = a & b
    union = carry
    s = a ^ b
    while carry:
        carry = (carry << 1) & MASK32
        newcarry = s & carry
        union |= newcarry
        s ^= carry
        carry = newcarry
    return union

def words_from_block(block: bytes) -> List[int]:
    return list(struct.unpack(">16I", block))

def expand_schedule(w16: List[int]) -> List[int]:
    W = list(w16)
    for t in range(16, 64):
        W.append(u32(sigma1(W[t - 2]) + W[t - 7] + sigma0(W[t - 15]) + W[t - 16]))
    return W

def pad_sha256(msg: bytes) -> bytes:
    bit_len = len(msg) * 8
    out = msg + b"\x80"
    while len(out) % 64 != 56:
        out += b"\x00"
    out += struct.pack(">Q", bit_len)
    return out

def dbl_sha256_hex_display(msg: bytes) -> str:
    return hashlib.sha256(hashlib.sha256(msg).digest()).digest()[::-1].hex()

@dataclass
class RoundTrace:
    t: int
    a: int
    b: int
    c: int
    d: int
    e: int
    f: int
    g: int
    h: int
    Wt: int
    T1: int
    T2: int
    stage_carries: Tuple[int, int, int, int]
    stage_mask_hw: Tuple[int, int, int, int]
    h_hw: int

def compress_block_trace(block: bytes, state: List[int]) -> Dict[str, object]:
    W = expand_schedule(words_from_block(block))
    a, b, c, d, e, f, g, h = state
    traces: List[RoundTrace] = []

    for t in range(64):
        s1 = Sigma1(e)
        chv = ch(e, f, g)

        sA, c1 = add32(h, s1)
        sB, c2 = add32(sA, chv)
        sC, c3 = add32(sB, K[t])
        T1, c4 = add32(sC, W[t])

        m1 = carry_mask_add(h, s1)
        m2 = carry_mask_add(sA, chv)
        m3 = carry_mask_add(sB, K[t])
        m4 = carry_mask_add(sC, W[t])

        T2 = u32(Sigma0(a) + maj(a, b, c))

        traces.append(
            RoundTrace(
                t=t, a=a, b=b, c=c, d=d, e=e, f=f, g=g, h=h,
                Wt=W[t], T1=T1, T2=T2,
                stage_carries=(c1, c2, c3, c4),
                stage_mask_hw=(hw(m1), hw(m2), hw(m3), hw(m4)),
                h_hw=hw(h),
            )
        )

        new_a = u32(T1 + T2)
        new_e = u32(d + T1)
        a, b, c, d, e, f, g, h = new_a, a, b, c, new_e, e, f, g

    state_out = [u32(state[i] + v) for i, v in enumerate([a, b, c, d, e, f, g, h])]
    return {
        "W": W,
        "traces": traces,
        "working_final": [a, b, c, d, e, f, g, h],
        "state_out": state_out,
    }

def sha256_trace(msg: bytes) -> List[Dict[str, object]]:
    padded = pad_sha256(msg)
    blocks = [padded[i:i+64] for i in range(0, len(padded), 64)]
    state = H0[:]
    out = []
    for idx, block in enumerate(blocks):
        step = compress_block_trace(block, state)
        out.append({
            "block_index": idx,
            "block": block,
            "init_state": state[:],
            **step,
        })
        state = step["state_out"]
    return out

@dataclass
class RoundGeometry:
    t: int
    stage_carries: Tuple[int, int, int, int]
    stage_mask_hw: Tuple[int, int, int, int]
    h_hw: int

@dataclass
class GeometryBundle:
    rounds: Dict[int, RoundGeometry]

def build_geometry_bundle(traces: List[RoundTrace]) -> GeometryBundle:
    return GeometryBundle(
        rounds={
            rt.t: RoundGeometry(rt.t, rt.stage_carries, rt.stage_mask_hw, rt.h_hw)
            for rt in traces
        }
    )

def reverse_step_from_next(next_state: List[int], t: int, W_guess: int) -> Dict[str, object]:
    a1, b1, c1, d1, e1, f1, g1, h1 = next_state

    a_t = b1
    b_t = c1
    c_t = d1
    e_t = f1
    f_t = g1
    g_t = h1

    T2 = u32(Sigma0(a_t) + maj(a_t, b_t, c_t))
    T1 = u32(a1 - T2)
    d_t = u32(e1 - T1)

    s1 = Sigma1(e_t)
    chv = ch(e_t, f_t, g_t)
    const_tail = u32(s1 + chv + K[t])
    h_t = u32(T1 - const_tail - W_guess)

    sA, c1 = add32(h_t, s1)
    sB, c2 = add32(sA, chv)
    sC, c3 = add32(sB, K[t])
    T1_check, c4 = add32(sC, W_guess)

    m1 = carry_mask_add(h_t, s1)
    m2 = carry_mask_add(sA, chv)
    m3 = carry_mask_add(sB, K[t])
    m4 = carry_mask_add(sC, W_guess)

    return {
        "state": [a_t, b_t, c_t, d_t, e_t, f_t, g_t, h_t],
        "stage_carries": (c1, c2, c3, c4),
        "stage_mask_hw": (hw(m1), hw(m2), hw(m3), hw(m4)),
        "h_hw": hw(h_t),
        "T1_check": T1_check,
        "T1": T1,
        "T2": T2,
    }

def geometry_score(bundle: GeometryBundle, next_state: List[int], t: int, guess: int) -> Dict[str, int]:
    obs = bundle.rounds[t]
    pred = reverse_step_from_next(next_state, t, guess)

    carry_mismatch = sum(int(a != b) for a, b in zip(pred["stage_carries"], obs.stage_carries))
    mask_hw_error = sum(abs(a - b) for a, b in zip(pred["stage_mask_hw"], obs.stage_mask_hw))
    h_hw_error = abs(pred["h_hw"] - obs.h_hw)
    total = 5 * carry_mismatch + mask_hw_error + h_hw_error
    return {
        "score": total,
        "carry_mismatch": carry_mismatch,
        "mask_hw_error": mask_hw_error,
        "h_hw_error": h_hw_error,
    }

def sample_probe(bundle: GeometryBundle, next_state: List[int], t: int, true_word: int, samples: int = 5000, seed: int = 2026) -> Dict[str, object]:
    rng = random.Random(seed)
    rows = []
    for _ in range(samples):
        g = rng.getrandbits(32)
        rows.append((geometry_score(bundle, next_state, t, g)["score"], g))
    rows.sort()
    true_stats = geometry_score(bundle, next_state, t, true_word)
    true_pair = (true_stats["score"], true_word)
    rank = 1 + sum(1 for row in rows if row < true_pair)
    return {
        "true_word": true_word,
        "true_stats": true_stats,
        "rank_among_random": rank,
        "best_random": rows[:10],
        "median_random_score": statistics.median([s for s, _ in rows]),
        "min_random_score": rows[0][0],
    }

def as_hex32(x: int) -> str:
    return f"0x{x:08x}"


## Real Bitcoin data used

### 1) Genesis block (block 0)
Header fields from the published raw genesis block:
- version = `01000000`
- previous block hash = `00...00`
- merkle root = `3ba3edfd...4b1e5e4a`
- time = `29ab5f49`
- bits = `ffff001d`
- nonce = `1dac2b7c`

### 2) Mainnet block 328,734
This is the exact 80-byte header shown in the official Bitcoin developer reference example:
- version = `02000000`
- previous block hash = `b6ff0b1b...00000000`
- merkle root = `9d10aa52...aab31471`
- time = `24d95a54`
- bits = `30c31b18`
- nonce = `fe9f0864`

### 3) Live network context
A recent mainnet block from mempool.space:
- height 943,411
- hash `00000000000000000000bef7f0870c24f2962cf83949e96c7288cf30f0d74bf0`
- timestamp `2026-04-02T21:02:41Z`
- version `0x21880000`
- bits `0x17021a91`
- nonce `0x2d641708`
- merkle root `be0d649734c927c090bbc7f87962d87c1d28fb86b34fa772f245e8fa91faf9de`

The live block is included as current context; the fully reconstructable probe runs below use the two headers for which all 80 bytes are present in-source.


In [2]:
GENESIS_HEADER_HEX = (
    "01000000"
    + "00" * 32
    + "3ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4a"
    + "29ab5f49"
    + "ffff001d"
    + "1dac2b7c"
)

BLOCK_328734_HEADER_HEX = (
    "02000000"
    "b6ff0b1b1680a2862a30ca44d346d9e8"
    "910d334beb48ca0c0000000000000000"
    "9d10aa52ee949386ca9385695f04ede2"
    "70dda20810decd12bc9b048aaab31471"
    "24d95a54"
    "30c31b18"
    "fe9f0864"
)

REAL_HEADERS = {
    "genesis": {
        "height": 0,
        "known_hash": "000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f",
        "header": bytes.fromhex(GENESIS_HEADER_HEX),
    },
    "block_328734": {
        "height": 328734,
        "known_hash": "000000000000000009a11b3972c8e532fe964de937c9e0096b43814e67af3728",
        "header": bytes.fromhex(BLOCK_328734_HEADER_HEX),
    },
}

for name, item in REAL_HEADERS.items():
    calc = dbl_sha256_hex_display(item["header"])
    print(name, len(item["header"]), calc, calc == item["known_hash"])


genesis 80 000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f True
block_328734 80 000000000000000009a11b3972c8e532fe964de937c9e0096b43814e67af3728 True


In [3]:
def second_block_seed_words(header: bytes) -> List[int]:
    blocks = [pad_sha256(header)[i:i+64] for i in range(0, len(pad_sha256(header)), 64)]
    return [int.from_bytes(blocks[1][i:i+4], "big") for i in range(0, 16, 4)]

for name, item in REAL_HEADERS.items():
    seeds = second_block_seed_words(item["header"])
    print(name)
    print("  W[0] merkle tail =", as_hex32(seeds[0]))
    print("  W[1] time        =", as_hex32(seeds[1]))
    print("  W[2] bits        =", as_hex32(seeds[2]))
    print("  W[3] nonce       =", as_hex32(seeds[3]))


genesis
  W[0] merkle tail = 0x4b1e5e4a
  W[1] time        = 0x29ab5f49
  W[2] bits        = 0xffff001d
  W[3] nonce       = 0x1dac2b7c
block_328734
  W[0] merkle tail = 0xaab31471
  W[1] time        = 0x24d95a54
  W[2] bits        = 0x30c31b18
  W[3] nonce       = 0xfe9f0864


## Probe target

For each real header we use:

- the **second compression block of the first SHA-256 pass**
- the **last round** `t = 63`
- the same **one-round tension probe** developed earlier

This is the most relevant place for Bitcoin because the second compression block is exactly where the mining fields (`time`, `bits`, `nonce`) enter the first SHA pass as seed words.


In [4]:
def analyze_header(header: bytes, known_hash: str) -> Dict[str, object]:
    pass1 = sha256_trace(header)
    assert len(pass1) == 2, "Bitcoin 80-byte header should produce exactly two compression blocks in the first SHA pass."

    mining_block = pass1[1]
    mining_bundle = build_geometry_bundle(mining_block["traces"])
    mining_next_state = mining_block["working_final"]
    mining_true_word = mining_block["W"][63]
    mining_probe = sample_probe(mining_bundle, mining_next_state, t=63, true_word=mining_true_word, samples=5000, seed=2026)

    first_digest = hashlib.sha256(header).digest()
    pass2 = sha256_trace(first_digest)
    assert len(pass2) == 1

    outer_block = pass2[0]
    outer_bundle = build_geometry_bundle(outer_block["traces"])
    outer_next_state = outer_block["working_final"]
    outer_true_word = outer_block["W"][63]
    outer_probe = sample_probe(outer_bundle, outer_next_state, t=63, true_word=outer_true_word, samples=5000, seed=2026)

    return {
        "block_hash": known_hash,
        "mining_block": mining_block,
        "mining_probe": mining_probe,
        "outer_block": outer_block,
        "outer_probe": outer_probe,
    }

analyses = {name: analyze_header(item["header"], item["known_hash"]) for name, item in REAL_HEADERS.items()}
list(analyses.keys())


['genesis', 'block_328734']

In [5]:
summary_rows = []
for name, result in analyses.items():
    summary_rows.append({
        "name": name,
        "block_hash": result["block_hash"],
        "first_pass_block2_true_W63": as_hex32(result["mining_probe"]["true_word"]),
        "first_pass_block2_true_score": result["mining_probe"]["true_stats"]["score"],
        "first_pass_block2_rank": result["mining_probe"]["rank_among_random"],
        "first_pass_block2_best_random_score": result["mining_probe"]["min_random_score"],
        "outer_pass_true_W63": as_hex32(result["outer_probe"]["true_word"]),
        "outer_pass_true_score": result["outer_probe"]["true_stats"]["score"],
        "outer_pass_rank": result["outer_probe"]["rank_among_random"],
        "outer_pass_best_random_score": result["outer_probe"]["min_random_score"],
    })

summary_rows


[{'name': 'genesis',
  'block_hash': '000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f',
  'first_pass_block2_true_W63': '0x86b0b8d5',
  'first_pass_block2_true_score': 0,
  'first_pass_block2_rank': 1,
  'first_pass_block2_best_random_score': 3,
  'outer_pass_true_W63': '0x9bd5aca3',
  'outer_pass_true_score': 0,
  'outer_pass_rank': 1,
  'outer_pass_best_random_score': 3},
 {'name': 'block_328734',
  'block_hash': '000000000000000009a11b3972c8e532fe964de937c9e0096b43814e67af3728',
  'first_pass_block2_true_W63': '0xa572aedd',
  'first_pass_block2_true_score': 0,
  'first_pass_block2_rank': 1,
  'first_pass_block2_best_random_score': 4,
  'outer_pass_true_W63': '0xd496b428',
  'outer_pass_true_score': 0,
  'outer_pass_rank': 1,
  'outer_pass_best_random_score': 3}]

### Reading the summary

- `true_score = 0` means the real word is a perfect fit to the exported side-geometry for that round.
- `rank = 1` means the true word was better than all 5000 sampled random guesses.
- `best_random_score > 0` means the hot/cold field is real: random guesses do not flatten onto the same value.
- `best_random_score` being **small** means there are near-misses. The probe is informative, not unique.


In [6]:
for name, result in analyses.items():
    print("=" * 88)
    print(name, "->", result["block_hash"])
    print("- first SHA pass, block 2, t=63")
    print("  true W[63]         :", as_hex32(result["mining_probe"]["true_word"]))
    print("  true score         :", result["mining_probe"]["true_stats"]["score"])
    print("  rank among 5000    :", result["mining_probe"]["rank_among_random"])
    print("  best random scores :", [(score, as_hex32(word)) for score, word in result["mining_probe"]["best_random"][:5]])
    print("- second SHA pass, t=63")
    print("  true W[63]         :", as_hex32(result["outer_probe"]["true_word"]))
    print("  true score         :", result["outer_probe"]["true_stats"]["score"])
    print("  rank among 5000    :", result["outer_probe"]["rank_among_random"])
    print("  best random scores :", [(score, as_hex32(word)) for score, word in result["outer_probe"]["best_random"][:5]])


genesis -> 000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f
- first SHA pass, block 2, t=63
  true W[63]         : 0x86b0b8d5
  true score         : 0
  rank among 5000    : 1
  best random scores : [(3, '0x7f74a135'), (4, '0x86b8b9d3'), (4, '0x8a42fdf1'), (4, '0x8eb5140f'), (4, '0x8ff0624a')]
- second SHA pass, t=63
  true W[63]         : 0x9bd5aca3
  true score         : 0
  rank among 5000    : 1
  best random scores : [(3, '0x8ff59cc6'), (3, '0xbf47412e'), (3, '0xf39623eb'), (4, '0x69f67a92'), (4, '0x7515d9ef')]
block_328734 -> 000000000000000009a11b3972c8e532fe964de937c9e0096b43814e67af3728
- first SHA pass, block 2, t=63
  true W[63]         : 0xa572aedd
  true score         : 0
  rank among 5000    : 1
  best random scores : [(4, '0x91b0ea5c'), (5, '0xb0c1a87c'), (6, '0x80fa6603'), (6, '0x9cbdaa2c'), (6, '0xa02fb007')]
- second SHA pass, t=63
  true W[63]         : 0xd496b428
  true score         : 0
  rank among 5000    : 1
  best random scores : [(3, '0xf714c46

## What this means for Bitcoin

The real result is not “Bitcoin is inverted.”

The real result is narrower and cleaner:

1. On **real Bitcoin mainnet headers**, the probe still produces a **hot/cold field**.
2. The true word at the tested round gets **score 0**.
3. In these two real-header runs, the true word ranks **#1 out of 5000 sampled random guesses**.
4. But low-scoring false candidates remain, so a **single local probe is not unique**.

That makes the Bitcoin result consistent with the earlier SHA notebook:
- the side-channel geometry is **measurable**
- the probe is **informative**
- one round alone is **not enough** for full recovery

So the next mechanically honest step is still the same:
**couple several reverse rounds together** and require a candidate to stay cold across a chain, not just one local cell.


## Live block context snapshot

Recent mainnet block seen in search results:
- **Height:** 943,411
- **Hash:** `00000000000000000000bef7f0870c24f2962cf83949e96c7288cf30f0d74bf0`
- **Timestamp:** `2026-04-02T21:02:41Z`
- **Version:** `0x21880000`
- **Bits:** `0x17021a91`
- **Nonce:** `0x2d641708`
- **Merkle root:** `be0d649734c927c090bbc7f87962d87c1d28fb86b34fa772f245e8fa91faf9de`

That live block is included to show the current proof-of-work field values are structurally the same kind of object as the historical headers analyzed above.
